# Récupération des factures à traiter dans le dossier factures

# Définition du modèle LLM

In [36]:

from datetime import date
import os
from typing import Literal, Union

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, SecretStr

api_key=APIM_OPENAI_API_KEY
feature="my-feature"

def _build_apim_headers(
    *,
    feature: str,
    api_key: str | None = None,
) -> dict[str, str]:
    headers: dict[str, str] = {
        "api-key": api_key or "",
        "owner": "HAMILTON",
        "feature": feature,
    }

    return headers

from datetime import date
from typing import Literal
from pydantic import BaseModel, Field, model_validator
import os

class DocumentInfo(BaseModel):
    """Schéma unifié pour facture OU bon de livraison."""
    type_document: Literal["facture", "bon_livraison"]
    
    # Champs communs
    numero_facture: str | None = None
    numero_bon_livraison: str | None = None
    date_emission: date | None = None
    date_livraison: date | None = None
    date_paiement_prevue: date | None = None
    montant_total: float | None = None
    nom_fournisseur: str | None = None
    bons_livraisons: list[str] = Field(default_factory=list)
    prix_HT_5_5pct: float | None = None
    prix_HT_10pct: float | None = None
    prix_HT_20pct: float | None = None
    
    @model_validator(mode='after')
    def validate_coherence(self):
        if self.type_document == "facture":
            if not self.numero_facture:
                raise ValueError("Une facture doit avoir un numero_facture")
        elif self.type_document == "bon_livraison":
            if not self.numero_bon_livraison:
                raise ValueError("Un bon de livraison doit avoir un numero_bon_livraison")
        return self

llm=ChatOpenAI(
        model="gpt-5.1-2025-11-13",
        api_key=SecretStr(api_key),
        base_url=APIM_OPENAI_BASE_URL,
        default_headers=_build_apim_headers(feature=feature, api_key=api_key),
        use_responses_api=False,
        streaming=False,
        reasoning_effort="low",
        temperature=1.0,
        max_retries=3,
        max_completion_tokens=1024,
        verbose=True,
    ).with_structured_output(
    DocumentInfo,
    method="json_schema",
    strict=False,
    include_raw=False,
)

# Transformation des PDF en markdown pour lecture LLM

In [37]:
from markitdown import MarkItDown
from openai import OpenAI

llm_client = OpenAI(
    api_key=api_key,
    base_url=APIM_OPENAI_BASE_URL,
    default_headers=_build_apim_headers(feature=feature, api_key=api_key),
)

md = MarkItDown(
    enable_plugins=True,
    llm_client=llm_client,
    llm_model="gpt-5.1-2025-11-13",
)

def load_and_split_pdf(file_path: str) -> str:
    # Conservé pour compatibilité: extraction texte via MarkItDown uniquement.
    return md.convert(file_path).text_content

# Classifieurs

In [38]:
import os
import re
from calendar import monthrange
from datetime import date, timedelta
from pprint import pprint

def _is_valid_business_date(value: date | None) -> bool:
    return isinstance(value, date) and 2020 <= value.year <= 2100

def _extract_date_candidates(text: str) -> list[date]:
    candidates: list[date] = []
    seen = set()

    for d, m, y in re.findall(r"\b(\d{2})[/-](\d{2})[/-](\d{4})\b", text):
        try:
            dt = date(int(y), int(m), int(d))
            if 2020 <= dt.year <= 2100 and dt not in seen:
                seen.add(dt)
                candidates.append(dt)
        except ValueError:
            pass

    for y, m, d in re.findall(r"\b(20\d{2})(\d{2})(\d{2})\b", text):
        try:
            dt = date(int(y), int(m), int(d))
            if 2020 <= dt.year <= 2100 and dt not in seen:
                seen.add(dt)
                candidates.append(dt)
        except ValueError:
            pass

    return sorted(candidates)

def _extract_date_from_filename(filename: str) -> date | None:
    match = re.search(r"(20\d{2})(\d{2})(\d{2})", filename)
    if not match:
        return None
    y, m, d = match.groups()
    try:
        return date(int(y), int(m), int(d))
    except ValueError:
        return None

def _infer_due_date(emission: date | None, text: str) -> date | None:
    if not emission:
        return None

    lower = text.lower()
    days_match = re.search(r"\b(\d{1,3})\s*jours?\b", lower)
    delay_days = int(days_match.group(1)) if days_match else None

    if "fin de mois" in lower and delay_days is not None:
        end_of_month = date(
            emission.year,
            emission.month,
            monthrange(emission.year, emission.month)[1],
        )
        return end_of_month + timedelta(days=delay_days)

    if delay_days is not None:
        return emission + timedelta(days=delay_days)

    return None

def classify_document(text: str, filename: str) -> str:
    lower = text.lower()
    head = lower[:8000]

    invoice_score = 0
    delivery_score = 0

    if re.search(r"\bfacture\s*(n|no|num|numero|#|:)", head):
        invoice_score += 3
    if re.search(r"\bdate\s+facture\b", head):
        invoice_score += 2
    if re.search(r"\binvoice\b", head):
        invoice_score += 2
    if "facture" in head:
        invoice_score += 1

    delivery_markers = [
        "bon de livraison",
        "bon livraison",
        "bl n",
        "bl n°",
        "a livrer le",
        "ar cde",
        "accuse de reception",
        "accusé de reception",
    ]
    for marker in delivery_markers:
        if marker in head:
            delivery_score += 1

    filename_lower = filename.lower()
    if re.search(r"\bbl\b", filename_lower):
        delivery_score += 2
    if re.search(r"\bc\d{5,}\b", filename_lower):
        delivery_score += 2
    if re.search(r"\bf\d{5,}\b", filename_lower):
        invoice_score += 2

    if delivery_score >= 2 and delivery_score >= invoice_score + 1:
        return "bon_livraison"
    if invoice_score >= 2 and invoice_score >= delivery_score + 1:
        return "facture"

    if delivery_score > invoice_score:
        return "bon_livraison"
    if invoice_score > delivery_score:
        return "facture"

    return "inconnu"

def _normalize_invoice_dates(data: dict, text: str, filename: str) -> dict:
    candidates = _extract_date_candidates(text)
    file_date = _extract_date_from_filename(filename)

    emission = data.get("date_emission")
    if not _is_valid_business_date(emission):
        if file_date is not None:
            emission = file_date
        elif candidates:
            emission = candidates[0]
        else:
            emission = None
        data["date_emission"] = emission

    due = data.get("date_paiement_prevue")
    if not _is_valid_business_date(due):
        inferred = _infer_due_date(emission, text)
        if inferred is not None and _is_valid_business_date(inferred):
            due = inferred
        else:
            later_dates = [d for d in candidates if emission and d >= emission]
            due = later_dates[-1] if later_dates else None
        data["date_paiement_prevue"] = due

    return data


# Traitement automatique

In [39]:
factures_traitees = []
bons_livraison_traites = []

for file in os.listdir('factures'):
    if not file.lower().endswith('.pdf'):
        continue
    
    print(f"Processing file {file}")
    
    text = load_and_split_pdf('factures/' + file)  # Ta fonction existante
    document_type = classify_document(text, file)  # Ta fonction existante
    
    if document_type == "bon_livraison":
        # Prompt spécifique bon de livraison
        result = llm.invoke([
            ("system", """Extrais uniquement les informations du bon de livraison depuis ce texte.
Si une valeur est absente, retourne null.
Normalise les montants en nombres."""),
            ("user", f"Texte bon de livraison:\n\n{text}")
        ])
        
        data = result.model_dump() if hasattr(result, 'model_dump') else result
        data['document_type'] = 'bon_livraison'
        data['fichier_source'] = file
        bons_livraison_traites.append(data)
        pprint(data)
        print(f"Finished processing file {file}")
        continue
    
    # Facture (cas par défaut)
    result = llm.invoke([
        ("system", f"""Extrais uniquement les informations de facture depuis ce texte.
type_document doit être "facture".
Si une valeur est absente, retourne null.
Normalise les montants en nombres.
Si des bons de livraison sont présents dans la facture, renseigne-les dans la liste dédiée.

Règles strictes sur les dates:
- Date d'émission et date de paiement prévue doivent être comprises entre 2020 et 2100.
- Si la date n'est pas certaine, retourne null.
- Le format de date dans les factures est souvent dd/mm/yyyy.

Texte facture: {text}"""),
        ("user", text)
    ])
    
    data = result.model_dump() if hasattr(result, 'model_dump') else result
    data['document_type'] = 'facture'
    data['fichier_source'] = file
    data = normalize_invoice_dates(data, text, file)  # Ta fonction existante
    factures_traitees.append(data)
    pprint(data)
    print(f"Finished processing file {file}")

Processing file 01_AMBELYS-C215075.pdf


ValidationError: 1 validation error for DocumentInfo
  Value error, Un bon de livraison doit avoir un numero_bon_livraison [type=value_error, input_value={'type_document': 'bon_li...'prix_HT_20pct': 462.99}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [ ]:
factures_traitees

[{'numero_facture': '826802',
  'date_emission': datetime.date(2026, 3, 31),
  'date_paiement_prevue': datetime.date(2026, 4, 30),
  'montant_total': 1697.32,
  'nom_fournisseur': 'AMBELYS',
  'bons_livraisons': ['246054',
   '244254',
   '240084',
   '244255',
   '241709',
   '243338',
   '244256',
   '245433'],
  'prix_HT_5_5pct': None,
  'prix_HT_10pct': None,
  'prix_HT_20pct': 1414.43,
  'document_type': 'facture',
  'fichier_source': 'AMBELYS - 20260331 - 01_F826802 - 1697.32.pdf'},
 {'numero_facture': '1253776376',
  'date_emission': datetime.date(2026, 4, 15),
  'date_paiement_prevue': None,
  'montant_total': 927.32,
  'nom_fournisseur': 'SYSCO France',
  'bons_livraisons': [],
  'prix_HT_5_5pct': 868.11,
  'prix_HT_10pct': None,
  'prix_HT_20pct': 9.55,
  'document_type': 'facture',
  'fichier_source': 'SYSCO - 20260415 - 927.32.PDF'},
 {'numero_facture': '830166379183',
  'date_emission': datetime.date(2026, 4, 13),
  'date_paiement_prevue': datetime.date(2026, 5, 3),
  'mon

In [ ]:
bons_livraison_traites

[{'document_type': 'bon_livraison',
  'fichier_source': '01_AMBELYS-C215075.pdf'}]

In [ ]:
import polars as pl
pl.from_dicts(factures_traitees)

numero_facture,date_emission,date_paiement_prevue,montant_total,nom_fournisseur,bons_livraisons,prix_HT_5_5pct,prix_HT_10pct,prix_HT_20pct,document_type,fichier_source
str,date,date,f64,str,list[str],f64,null,f64,str,str
"""826802""",2026-03-31,2026-04-30,1697.32,"""AMBELYS""","[""246054"", ""244254"", … ""245433""]",null,null,1414.43,"""facture""","""AMBELYS - 20260331 - 01_F82680…"
"""1253776376""",2026-04-15,null,927.32,"""SYSCO France""",[],868.11,null,9.55,"""facture""","""SYSCO - 20260415 - 927.32.PDF"""
"""830166379183""",2026-04-13,2026-05-03,249.63,"""TERREAZUR AQUITAINE""","[""30992166""]",236.62,null,null,"""facture""","""TERRE AZUR - 20260413 - 249.63…"
